# HW2 - Pyro(NumPyro)를 활용한 베이지안 회귀모형 (변분추론)

**문제.** `immigrants` 자료에 대해 베이지안 회귀모형을 적용한다.

1. 변분추론(Stochastic Variational Inference, SVI)을 이용한다.
2. `wage`를 $Y$(종속변수)로, `sp`, `lit`, `n_live`를 $X$(독립변수)로 두고 회귀모형을 적합한다.

> 10강 '추가 예제(베이지안 회귀모형)'의 풀이 방식 — 데이터 표준화 → `regmodel`/수기 `guide` 정의 → `SVI` + `Adam` + `Trace_ELBO` 최적화 — 을 그대로 차용하여 독립변수 3개로 확장한다.

### Model
$$ y \sim N(X\beta,\; \tfrac{1}{\tau} I_{n}) $$
여기서 $y=(y_1,\cdots,y_N)^T$ 는 `wage`, $X \in \mathbb{R}^{N\times p}$ 는 `sp, lit, n_live`로 구성된 설계행렬이다.

### Priors
$$ a \sim N(0, 10^2),\quad b_{sp}, b_{lit}, b_{nlive} \sim N(0, 10^2),\quad \tau \sim Ga(0.01, 0.01) $$

### Variational distribution
$$ q_\lambda(\theta) = Ga(\tau \mid \alpha_\tau, \beta_\tau)\, N(a \mid \mu_a, \sigma_a^2) \prod_{j} N(b_j \mid \mu_{b_j}, \sigma_{b_j}^2) $$

변분추론의 수렴 안정성을 위해 변수들을 **표준화**한 뒤 적합한다 (추가 예제와 동일).

In [ ]:
import jax
import numpyro
import numpyro.distributions as dist
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# 데이터
dset = pd.read_csv('immigrants.csv', index_col=0)
dset.head()

In [ ]:
# 데이터 표준화 (평균 0, 표준편차 1)
standardize = lambda x: (x - x.mean()) / x.std()

dset['wageScaled'] = dset.wage.pipe(standardize)
dset['spScaled'] = dset.sp.pipe(standardize)
dset['litScaled'] = dset.lit.pipe(standardize)
dset['n_liveScaled'] = dset.n_live.pipe(standardize)

In [ ]:
# 모형정의 - likelihood, prior
def regmodel(sp=None, lit=None, n_live=None, wage=None):
    # prior
    a = numpyro.sample('a', dist.Normal(0, 10))            # 절편
    b_sp = numpyro.sample('b_sp', dist.Normal(0, 10))      # 기울기 sp
    b_lit = numpyro.sample('b_lit', dist.Normal(0, 10))    # 기울기 lit
    b_nlive = numpyro.sample('b_nlive', dist.Normal(0, 10))  # 기울기 n_live
    tausq = numpyro.sample('tausq', dist.Gamma(0.01, 0.01))
    sigma = jax.numpy.sqrt(1.0 / tausq)
    mu = a + b_sp * sp + b_lit * lit + b_nlive * n_live

    # likelihood
    with numpyro.plate('data', len(wage)):
        numpyro.sample('obs', dist.Normal(mu, sigma), obs=wage)

In [ ]:
# 변분분포 정의
def guide(sp=None, lit=None, n_live=None, wage=None):
    # tau
    alpha_tau = numpyro.param('alpha_tau', 1.0, constraint=dist.constraints.positive)
    beta_tau = numpyro.param('beta_tau', 1.0, constraint=dist.constraints.positive)
    numpyro.sample('tausq', dist.Gamma(alpha_tau, beta_tau))

    # a
    loc_a = numpyro.param('loc_a', 0.0)
    scale_a = numpyro.param('scale_a', 1.0, constraint=dist.constraints.positive)
    numpyro.sample('a', dist.Normal(loc_a, scale_a))

    # b_sp
    loc_sp = numpyro.param('loc_sp', 0.0)
    scale_sp = numpyro.param('scale_sp', 1.0, constraint=dist.constraints.positive)
    numpyro.sample('b_sp', dist.Normal(loc_sp, scale_sp))

    # b_lit
    loc_lit = numpyro.param('loc_lit', 0.0)
    scale_lit = numpyro.param('scale_lit', 1.0, constraint=dist.constraints.positive)
    numpyro.sample('b_lit', dist.Normal(loc_lit, scale_lit))

    # b_nlive
    loc_nlive = numpyro.param('loc_nlive', 0.0)
    scale_nlive = numpyro.param('scale_nlive', 1.0, constraint=dist.constraints.positive)
    numpyro.sample('b_nlive', dist.Normal(loc_nlive, scale_nlive))

In [ ]:
# 최적화
from numpyro.infer import SVI, Trace_ELBO
from numpyro.optim import Adam

# 최적화를 위한 Adam 옵티마이저 설정
optimizer = Adam(step_size=0.001)

# SVI 객체 생성
svi = SVI(regmodel, guide, optimizer, loss=Trace_ELBO())

# SVI 실행
n_steps = 10000
svi_state = svi.init(jax.random.PRNGKey(0),
                     sp=dset.spScaled.values,
                     lit=dset.litScaled.values,
                     n_live=dset.n_liveScaled.values,
                     wage=dset.wageScaled.values)

# 최적화 루프
elbo_values = []
for step in range(n_steps):
    svi_state, elbo = svi.update(svi_state,
                                 sp=dset.spScaled.values,
                                 lit=dset.litScaled.values,
                                 n_live=dset.n_liveScaled.values,
                                 wage=dset.wageScaled.values)
    elbo_values.append(-elbo)
    if step % 1000 == 0:
        print('step {}: ELBO = {}'.format(step, elbo))

In [ ]:
# 수렴성 판단
plt.figure(figsize=(10, 6))
plt.plot(elbo_values)
plt.title('ELBO over time')
plt.xlabel('Step')
plt.ylabel('ELBO')
plt.show()

In [ ]:
# 추론된 변분 모수
params = svi.get_params(svi_state)
print(params)

In [ ]:
# 회귀계수 사후 요약 (표준화 척도)
# 변분분포가 Normal이므로 95% 신용구간 = loc +/- 1.96*scale
names = ['a', 'b_sp', 'b_lit', 'b_nlive']
locs = ['loc_a', 'loc_sp', 'loc_lit', 'loc_nlive']
scales = ['scale_a', 'scale_sp', 'scale_lit', 'scale_nlive']

print('{:<10}{:>10}{:>10}{:>10}{:>10}'.format('param', 'mean', 'sd', '2.5%', '97.5%'))
for nm, lc, sc in zip(names, locs, scales):
    m = float(params[lc]); s = float(params[sc])
    print('{:<10}{:>10.4f}{:>10.4f}{:>10.4f}{:>10.4f}'.format(nm, m, s, m - 1.96 * s, m + 1.96 * s))

# 오차 분산/표준편차: tausq ~ Gamma(alpha, beta) => E[tausq]=alpha/beta, sigma=sqrt(1/tausq)
tausq_mean = float(params['alpha_tau'] / params['beta_tau'])
print('\nE[tausq] =', round(tausq_mean, 4), '  =>  sigma =', round(float(np.sqrt(1.0 / tausq_mean)), 4))

In [ ]:
# (참고) 계수를 원래 척도로 환산하여 해석
# 표준화: x' = (x - mx)/sx, y' = (y - my)/sy
# => b_orig_j = b_std_j * sy / sx_j ,  a_orig = my + sy*a_std - sum_j b_orig_j * mx_j
sy = dset.wage.std()
my = dset.wage.mean()
feat = {'sp': 'loc_sp', 'lit': 'loc_lit', 'n_live': 'loc_nlive'}

b_orig = {}
for col, key in feat.items():
    b_orig[col] = float(params[key]) * sy / dset[col].std()
a_orig = my + sy * float(params['loc_a']) - sum(b_orig[c] * dset[c].mean() for c in feat)

print('원래 척도 회귀식: wage =')
print('   {:.4f}'.format(a_orig))
for col in feat:
    print('   {:+.4f} * {}'.format(b_orig[col], col))

### 분석 결과

- **ELBO**가 안정적으로 수렴하여 변분추론이 정상적으로 이루어졌다.
- 표준화 척도 회귀계수의 사후평균과 95% 신용구간을 보면,
  - **`lit`(문해율)** 의 계수가 가장 크고 양(+)이며 신용구간이 0을 포함하지 않는다.
  - **`sp`** 의 계수도 양(+)이며 신용구간이 0을 포함하지 않아 임금에 유의하게 기여한다.
  - **`n_live`** 의 계수는 0에 가깝고 신용구간이 0을 포함하여, 임금에 대한 효과가 뚜렷하지 않다.
- 즉, 이민자 집단의 **임금(`wage`)은 문해율(`lit`)과 영어 사용 비율(`sp`)이 높을수록 높아지는 경향**이 있으며, 그중 `lit`의 영향이 가장 크다. `n_live`의 효과는 통계적으로 불분명하다.